# Imports

In [1]:
import sys
import torch
import os
import numpy as np
import nltk
sys.path.append('../..')
from nltk.corpus import stopwords
import pandas as pd
import plotly.express as px
from definitions import EXPERIMENTS_DIR
from typing import Optional, List, Dict, Tuple, Literal
from torch import Tensor
from src.commands.graph_learning.main import Trainer, Config, Graph
from src.commands.feature_extraction.graph_builder import EdgeTypes
from sklearn.cluster import KMeans
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from torch_geometric import utils as gnn_utils
from torch_scatter import scatter
from torch_geometric.data import HeteroData
from functools import cache
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import NearestNeighbors
from src import helpers

In [2]:
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /home/fellah-a/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/fellah-a/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /home/fellah-a/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/fellah-a/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [3]:
px.defaults.template = "seaborn"

In [4]:
def warn(*args, **kwargs):
    pass

import warnings
warnings.warn = warn

# Utils

In [5]:
def load_histories(path : str) -> pd.DataFrame:

    histories_dir = os.path.join(path, 'histories')
    histories = []

    for filename in os.listdir(histories_dir):
        filename = os.path.join(histories_dir, filename)
        df = pd.read_csv(filename)
        df['iteration'] = np.arange(len(df))
        histories.append(df)

    histories = pd.concat(histories)

    return histories #.groupby(by=['epoch', 'phase', 'config']).mean().reset_index()

In [6]:
def load_results(path : str) -> Dict[str, Dict[str, Tensor]]:

    results_dir = os.path.join(path, 'results')
    results = {}

    for filename in os.listdir(results_dir):
        result_path = os.path.join(results_dir, filename)
        exp = filename.split('.')[0].replace('predictions_', '')
        results[exp] = torch.load(result_path, map_location='cpu', weights_only=False)

    return results

In [7]:
@cache
def load_data(experiment : str) -> Tuple[Graph, HeteroData]:

    config = Config(experiment=experiment)
    trainer = Trainer(experiment=config.experiment, config=config.training_config, configs=['Attn'])

    data, graph = trainer.load_data(config=config.training_config, return_raw=True)

    return graph, data

In [8]:
@cache
def load_model(experiment : str, config_name : str) -> Tuple[Graph, HeteroData, torch.nn.Module]:

    config = Config(experiment=experiment)
    trainer = Trainer(experiment=config.experiment, config=config.training_config, configs=['Attn'])

    data, graph = load_data(experiment=experiment)

    model_config = config.training_config.configs[config_name]
    model = trainer.create_model(config=model_config, data=graph)

    print(model.load_state_dict(torch.load(model_config.model_path, weights_only=False)))

    return model

In [9]:
def get_dataframes(graph : Graph) -> List[pd.DataFrame]:

    articles_df = pd.DataFrame(data=list(map(lambda x : { 'id' : x.id,  **x.properties }, graph.get_nodes('Article').nodes_generator))).set_index('id')

    entities_df = pd.DataFrame(data=list(map(lambda x : { 'id' : x.id, **x.properties }, graph.get_nodes('Entity').nodes_generator)))
    
    triples_df = pd.DataFrame(data=list(map(lambda x : {
        "id" : x.id,
        "src" : x.src,
        "dst" : x.dst,
        "predicate" : x.properties['predicate']
    }, graph.get_edges(EdgeTypes.Entity_HASRELATIONWITH_Entity).edges_generator)))

    articles_df = articles_df \
        .join(pd.DataFrame(graph._edges[EdgeTypes.Article_HASKEYWORD_Keyword].edges_generator)[['src', 'dst']] \
        .groupby('src').agg(list)) \
        .rename(columns={'dst':'keywords'})

    return articles_df, entities_df, triples_df

In [10]:
def style_paper_plot(
    fig,
    *,
    width=950,
    height=600,
    font_family="Arial",
    font_size=14,
):
    fig.update_layout(
        width=width,
        height=height,
        font=dict(
            family=font_family,
            size=font_size,
        ),
        plot_bgcolor="white",
        paper_bgcolor="white",
        margin=dict(
            l=80,
            r=30,
            t=30,
            b=70,
        ),
    )

    fig.update_xaxes(
        showgrid=False,
        zeroline=False,
        ticks="outside",
        showline=True,
        linewidth=1,
        linecolor="black",
    )

    fig.update_yaxes(
        showgrid=True,
        gridcolor="lightgray",
        gridwidth=0.5,
        zeroline=False,
        ticks="outside",
        showline=True,
        linewidth=1,
        linecolor="black",
    )

    return fig

In [11]:
EXPERIMENT = "topic_modeling/pooling"
EXPERIMENT_PATH = os.path.join(EXPERIMENTS_DIR, EXPERIMENT)
config = Config(experiment=EXPERIMENT)

In [ ]:
pd.concat([
    load_histories(os.path.join(EXPERIMENTS_DIR, EXPERIMENT))
])

In [ ]:
history = load_histories(EXPERIMENT_PATH)
results = load_results(EXPERIMENT_PATH)

In [37]:
encoder_decoder_history = pd.read_csv('/home/fellah-a/Documents/Text2KG/experiments/topic_modeling/encoder_decoder/histories/history_baseline.csv')

In [44]:
encoder_decoder_history = encoder_decoder_history.rename(columns={
    "cosine_rec_loss__MAE" : "rec_loss__MAE",
    "cosine_rec_loss" : "rec_loss",
    "cosine_rec_loss__R2" : "rec_loss__R2"
})
encoder_decoder_history['config'] = 'Encoder Decoder'

In [39]:
history

,epoch,phase,loss,rec_loss__MAE,rec_loss__R2,rec_loss,config,iteration
0,0,train,0.870786,0.922865,-87.924889,0.870786,baseline,0
1,0,train,0.800565,0.844166,-8.950556,0.800565,baseline,1
2,0,train,0.891071,0.935295,-1.147517,0.891071,baseline,2
3,0,train,0.780599,0.811259,-3.651428,0.780599,baseline,3
4,0,train,0.836324,0.889662,-22.457430,0.836324,baseline,4
...,...,...,...,...,...,...,...,...
4595,49,val,0.013577,0.074089,0.080978,0.013577,baseline_v2,4595
4596,49,val,0.014331,0.076150,0.032583,0.014331,baseline_v2,4596
4597,49,val,0.013632,0.074104,0.078058,0.013632,baseline_v2,4597
4598,49,val,0.013661,0.074313,0.071383,0.013661,baseline_v2,4598


In [45]:
history = pd.concat([
    history, 
    encoder_decoder_history.rename(columns={
        "cosine_rec_loss__MAE" : "rec_loss__MAE",
        "cosine_rec_loss" : "rec_loss",
        "cosine_rec_loss__R2" : "rec_loss__R2"
    })
])

In [13]:
graph, data = load_data(experiment=EXPERIMENT)
# model = load_model(experiment=EXPERIMENT, config_name="baseline")

len(self.configs) = 1


/home/fellah-a/anaconda3/envs/text2kg-main/lib/python3.12/site-packages/torch_sparse/matmul.py:97: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:53.)
  C = torch.sparse.mm(A, B)


In [34]:
print(f"Number OF Articles : {data['Article'].x.size(0)}")
print(f"Number OF Entities : {data['Entity'].x.size(0)}")
print(f"Number OF Triples : {data['Entity', 'HASRELATIONWITH', 'Entity'].edge_index.size(-1)}")

Number OF Articles : 748
Number OF Entities : 6318
Number OF Triples : 7479


In [40]:
gnn_utils.degree(
    index=data['Entity', 'HASRELATIONWITH', 'Entity'].edge_index[1], 
    num_nodes=data['Entity'].x.size(0)
).mean()

tensor(1.1838)

In [41]:
gnn_utils.degree(
    index=data['Entity', 'HASRELATIONWITH', 'Entity'].edge_index[1], 
    num_nodes=data['Entity'].x.size(0)
).max()

tensor(30.)

In [14]:
fig = px.histogram(
    x=gnn_utils.degree(
        index=data['Entity', 'HASRELATIONWITH', 'Entity'].edge_index[1], 
        num_nodes=data['Entity'].x.size(0)
    ).numpy()
)
fig.update_layout(width=900, height=750)
fig.show()

In [15]:
helpers.torch_connected_components(
    edge_index=data['Entity', 'HASRELATIONWITH', 'Entity'].edge_index,
    num_nodes=data['Entity'].x.size(0)
).unique(return_counts=True)

(tensor([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,  13,
          14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,  26,  27,
          28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,  39,  40,  41,
          42,  43,  44,  45,  46,  47,  48,  49,  50,  51,  52,  53,  54,  55,
          56,  57,  58,  59,  60,  61,  62,  63,  64,  65,  66,  67,  68,  69,
          70,  71,  72,  73,  74,  75,  76,  77,  78,  79,  80,  81,  82,  83,
          84,  85,  86,  87,  88,  89,  90,  91,  92,  93,  94,  95,  96,  97,
          98,  99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111,
         112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125,
         126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139,
         140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153,
         154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167,
         168, 169, 170, 171, 172, 173, 174, 175, 176

In [16]:
articles_df, entities_df, triples_df = get_dataframes(graph)

In [48]:
history['config'].unique()

<StringArray>
[          'baseline',        'baseline_v4', 'baseline_fullbatch',
        'baseline_v3',        'baseline_v2',    'Encoder Decoder']
Length: 6, dtype: str

In [51]:
history[history['config'] == 'Encoder Decoder']

,epoch,phase,loss,rec_loss__MAE,rec_loss__R2,rec_loss,config,iteration
0,0,train,0.256536,0.497287,-33.970722,0.256536,Encoder Decoder,NaN
1,0,train,0.056793,0.208507,-6.673162,0.056793,Encoder Decoder,NaN
2,0,train,0.029575,0.137600,-2.976383,0.029575,Encoder Decoder,NaN
3,0,train,0.033193,0.149197,-3.518319,0.033193,Encoder Decoder,NaN
4,0,train,0.041185,0.170772,-4.568232,0.041185,Encoder Decoder,NaN
...,...,...,...,...,...,...,...,...
1495,49,val,0.003517,0.046446,0.526829,0.003517,Encoder Decoder,NaN
1496,49,val,0.003507,0.046315,0.524067,0.003507,Encoder Decoder,NaN
1497,49,val,0.003531,0.046489,0.521486,0.003531,Encoder Decoder,NaN
1498,49,val,0.003499,0.046262,0.532603,0.003499,Encoder Decoder,NaN


In [49]:
data = history.query("phase == 'val'") \
    .drop(columns='iteration') \
    .groupby(by=['phase', 'epoch', 'config']) \
    .mean().reset_index()
mapping = {
    "baseline" : "Activation=None,BatchNorm=False,Agg=Sum",
    "baseline_v2" : "Activation=relu,BatchNorm=False,Agg=Sum",
    "baseline_v3" : "Activation=None,BatchNorm=True,Agg=Sum",
    "baseline_v4" : "Activation=None,BatchNorm=False,Agg=Avg",
    "Encoder Decoder" : "Encoder Decoder"
}
data['config'] = data['config'].map(mapping.get)
data.head()

,phase,epoch,config,loss,rec_loss__MAE,rec_loss__R2,rec_loss
0,val,0,Encoder Decoder,0.069007,0.236613,-8.343877,0.069007
1,val,0,"Activation=None,BatchNorm=False,Agg=Sum",0.796630,0.822438,-5.354157,0.796630
2,val,0,NaN,0.894910,0.918182,-7.954056,0.894910
3,val,0,"Activation=relu,BatchNorm=False,Agg=Sum",0.796528,0.821477,-4.946493,0.796528
4,val,0,"Activation=None,BatchNorm=True,Agg=Sum",0.456056,0.657949,-29.557118,0.456056


In [52]:
data[data['epoch'] == 49]

,phase,epoch,config,loss,rec_loss__MAE,rec_loss__R2,rec_loss
294,val,49,Encoder Decoder,0.003512,0.046366,0.524392,0.003512
295,val,49,"Activation=None,BatchNorm=False,Agg=Sum",0.177756,0.327542,-0.028935,0.177756
296,val,49,NaN,0.891778,0.908099,-11.948380,0.891778
297,val,49,"Activation=relu,BatchNorm=False,Agg=Sum",0.013855,0.074779,0.067985,0.013855
298,val,49,"Activation=None,BatchNorm=True,Agg=Sum",0.903707,0.945903,-17555.584250,0.903707
299,val,49,"Activation=None,BatchNorm=False,Agg=Avg",0.290397,0.427789,-0.294471,0.290397


In [53]:
fig = px.line(data_frame=data, x='epoch', y='loss', color='config')
fig = style_paper_plot(fig=fig)
fig.update_layout(xaxis_title='Epoch', yaxis_title="Mean Squared Error (MSE)")
fig.show()
fig.write_image('loss_history.pdf')

In [19]:
fig = px.line(data_frame=data, x='epoch', y='rec_loss__MAE', color='config')
fig = style_paper_plot(fig=fig)
fig.update_layout(xaxis_title='Epoch', yaxis_title="Mean Absolute Error (MAE)")
fig.show()

In [20]:
data['rec_loss__R2'].max()

np.float64(0.07902152512384494)

In [32]:
data['rec_loss__R2'] = data['rec_loss__R2'].clip(lower=-20, upper=-10)    

In [33]:
fig = px.line(data_frame=data, x='epoch', y='rec_loss__R2', color='config')
fig = style_paper_plot(fig=fig)
fig.update_yaxes(range=[-20, data['rec_loss__R2'].max().item()])
fig.update_layout(xaxis_title='Epoch', yaxis_title="R2 Score")
fig.show()